In [1]:
!pip install --upgrade pip
!pip install torch-summary
!pip install arabert transformers pytorch_lightning

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 6.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.0/185.0 kB 16.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for emoji: filename=emoji-1.4.2-py3-none-any.whl size=186468 sha256=4df3c0973332093ca6fb511f13123dcd2e56913d7ad773262567e8fb26821bc3
  Stored in directory: /root/.cache/pip/wheels/10/f0/fd/4813b1177405693e8da9cdea839f0fb64fde161380e058c827
Successfully built emoji
  Attempting uninstall: emoji
    Found existing installation: emoji 2.2.0
    Uninstalling emoji-2.2.0:
      Successfully uninstalled emoji-2.2.0


In [ ]:
# !pip install install-jdk

# !pip install https://storage.googleapis.com/tpu-pytorch/wheels/tpuvm/torch_xla-2.0-cp38-cp38-linux_x86_64.whl

In [2]:
!git clone https://github.com/ARBML/rouge_score_ar
%cd rouge_score_ar
!pip install -r requirements.txt
!pip install -e .
%cd ..

Cloning into 'rouge_score_ar'...
remote: Enumerating objects: 40, done.
remote: Counting objects: 100% (40/40), done.
remote: Compressing objects: 100% (30/30), done.
remote: Total 40 (delta 10), reused 30 (delta 7), pack-reused 0
Receiving objects: 100% (40/40), 25.95 KiB | 5.19 MiB/s, done.
Resolving deltas: 100% (10/10), done.
/kaggle/working/rouge_score_ar
Obtaining file:///kaggle/working/rouge_score_ar
  Preparing metadata (setup.py) ... done
  Running setup.py develop for rouge-score-ar
/kaggle/working


In [ ]:
!ls 

In [ ]:
!mkdir 'data'
!mkdir 'data/train'
!mkdir 'data/dev'

In [ ]:
# import jdk
# from jdk.enums import OperatingSystem, Architecture

# jdk.install('11', operating_system=OperatingSystem.LINUX)

# import os

# os.environ['JAVA_HOME'] = '/root/.jdk/jdk-11.0.19+7'
# os.environ['PATH'] = f"{os.environ.get('PATH')}:{os.environ.get('JAVA_HOME')}/bin"

In [ ]:
# !java -version

In [ ]:
# !rm -r lightning_logs/

In [ ]:
# import os
# os.environ.pop('PJRT_DEVICE', None)


In [ ]:
from arabert.aragpt2.grover.modeling_gpt2 import GPT2LMHeadModel
from transformers import GPT2TokenizerFast

In [ ]:
MODEL_NAME='aubmindlab/aragpt2-base'

GPT_tokenizer = GPT2TokenizerFast.from_pretrained(MODEL_NAME)
# m = GPT2LMHeadModel.from_pretrained(MODEL_NAME, add_cross_attention=True, bad_words_ids=GPT_tokenizer.pad_token_id, decoder_start_token_id=GPT_tokenizer.)

In [ ]:
GPT_tokenizer.add_special_tokens( {'unk\_token': '<UNK>', 'sep\_token': '<SEP>', 'pad\_token': '<PAD>', 'cls\_token': '<CLS>'})

In [ ]:
GPT_tokenizer('أحمد يلعب الكره كل مساء', padding='max_length', max_length=100,)

In [ ]:
%cd rougecross_attention_hidden_size_score_ar
from rouge_score import rouge_scorer
%cd ..
import torch
from torchsummary import summary

from transformers import AutoTokenizer, AutoModel, BertLMHeadModel
from arabert.preprocess import ArabertPreprocessor


import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import pytorch_lightning as pl

from pytorch_lightning.callbacks import ModelCheckpoint, StochasticWeightAveraging
from pytorch_lightning.callbacks import  RichProgressBar
from pytorch_lightning.callbacks.progress.rich_progress import RichProgressBarTheme

import pandas as pd
import os
import copy
from tqdm import tqdm
import numpy as np
import pickle
import random

In [ ]:
def avg_rough(preds, target):
    metric = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'])
    scores = list(map(lambda x: metric.score(*x), zip(preds, target)))
    sum_rough = dict()
    max_len = len(scores)
    for item in scores:
        for key, score in item.items():
            if key not in sum_rough.keys():
                sum_rough[key] = dict()
                sum_rough[key]['fmeasure'] = score.fmeasure
                sum_rough[key]['recall'] = score.recall
                sum_rough[key]['precision'] = score.precision
            else:
                sum_rough[key]['fmeasure'] += score.fmeasure
                sum_rough[key]['recall'] += score.recall
                sum_rough[key]['precision'] += score.precision
            
    for key, score in sum_rough.items():
        for metric, value in score.items():
            sum_rough[key][metric] = sum_rough[key][metric] / max_len
            
        sum_rough[key] = sum_rough[key]['fmeasure']
    
    return sum_rough
        

In [ ]:
class Data(Dataset):
    def __init__(self, data, batch_size, in_size=512, out_size=100):
        self.data_values = data
        self.batch_size = batch_size
        self.in_size = in_size
        self.out_size = out_size

    def __getitem__(self, idx):
        path = self.data_values[idx]
        
        with open(path, 'rb') as f:
            batch = pickle.load(f)
            
        r = np.random.randint(0, 12, (self.batch_size,)).tolist()
        
        for k, v in batch.items():
            if k != 'label_out':
                for kk, vv in v.items():
                    vv.data = vv[r]
            else:
                v.data = v[r]
        
        return batch['seq_in'], (batch['label_in'], batch['label_out'])

    def __len__(self):
        return len(self.data_values)

In [ ]:
class EncoderDecoder4Summary(pl.LightningModule):
    def __init__(self, model_name, pad_token_id, max_position_embeddings, decoder=None, ckp=None, lr=1e-4):
        super().__init__()
        
        self.save_hyperparameters()
        
        self.encoder = AutoModel.from_pretrained(model_name, pad_token_id=self.hparams.pad_token_id, max_position_embeddings=max_position_embeddings, ignore_mismatched_sizes=True, position_embedding_type='relative')
        self.encoder.pooler = None
        self.decoder = BertLMHeadModel.from_pretrained(model_name, pad_token_id=self.hparams.pad_token_id, is_decoder=True, add_cross_attention=True, position_embedding_type='relative')
        self = self.cpu()
        
        if ckp is not None:
            with torch.no_grad():
                ckp = torch.load(ckp, map_location='cpu',)['state_dict']
                for n, p in self.encoder.named_parameters():
                    if p.size() != ckp[f'encoder.{n}'].size():
                        print('not intialize layer', n , 'from checkpoint')
                        continue
                    p.data = ckp[f'encoder.{n}'].cpu()
                for n, p in self.decoder.named_parameters():
                    if p.size() != ckp[f'decoder.{n}'].size():
                        print('not intialize layer', n , 'from checkpoint')
                        continue
                    p.data = ckp[f'decoder.{n}'].cpu()

            print('checkpoint loaded succfflly')
            
        self.train()
        for n, p in self.encoder.named_parameters():
            if False:
                p.requires_grad_(True)
            else:
                p.requires_grad_(False)
                
        for n, p in self.decoder.named_parameters():
    
#             if 'position' in n or 'word_embeddings' in n:
#                 p.requires_grad_(False)
#             else:
            p.requires_grad_(True)
                
    def forward(self, seq, label):

        context = self.encoder(**seq).last_hidden_state
        out = self.decoder(**label, encoder_hidden_states=context, encoder_attention_mask=seq['attention_mask'])
        return out

    def training_step(self, batch, batch_idx):
        seq, (label_in, label_out) = batch
        out = self(seq, label_in)
        loss = F.cross_entropy(out.logits.transpose(2, 1), label_out, ignore_index=self.encoder.config.pad_token_id)

        self.log('Loss', loss, on_step=True, on_epoch=False, sync_dist=True, prog_bar=True,)
        return loss

    def validation_step(self, batch, batch_idx):
        seq, (label_in, label_out) = batch

        out = self(seq, label_in)

        loss = F.cross_entropy(out.logits.transpose(2, 1), label_out, ignore_index=self.encoder.config.pad_token_id)
        preds  = torch.argmax(out.logits, -1)
        preds  = bert_tokenizer.batch_decode(preds, skip_special_tokens=True)
        preds = list(map(bert_prep.unpreprocess, preds))
        
        target = bert_tokenizer.batch_decode(label_out, skip_special_tokens=True)
        target = list(map(bert_prep.unpreprocess, target))
        rough = avg_rough(preds, target)
        rough['val_loss'] = loss
        self.log_dict(rough, on_epoch=True, sync_dist=True, prog_bar=True)


    def configure_optimizers(self):
        pp = [p for p in self.parameters() if p.requires_grad]
        opt = torch.optim.AdamW(pp, lr=self.hparams.lr, weight_decay=0.1)
        return opt
        scheduler = {
            "scheduler": CosineAnnealingWarmRestarts(optimizer, T_0=6, T_mult=2, eta_min=0, verbose=True),
            "interval": "epoch",
            "frequency": 1,}
        return [optimizer], [scheduler]


In [ ]:
ckp = ModelCheckpoint(every_n_train_steps=100, save_last=True, auto_insert_metric_name=False)
progress_bar=RichProgressBar(theme=RichProgressBarTheme(description="green_yellow",
                                                          progress_bar="green1",
                                                          progress_bar_finished="green1",
                                                          progress_bar_pulse="#6206E0",
                                                          batch_progress="green_yellow",
                                                          time="grey82",
                                                          processing_speed="grey82",
                                                          metrics="green1"))

In [ ]:
model_name = "aubmindlab/bert-base-arabertv2"
ckp_path = '/kaggle/input/salah-checkpoint/lightning_logs/version_0/checkpoints/last.ckpt' #change to None for fresh model

bert_prep = ArabertPreprocessor(model_name=model_name)
bert_tokenizer = AutoTokenizer.from_pretrained(model_name,)

model = EncoderDecoder4Summary(model_name, bert_tokenizer.pad_token_id, 1000, ckp=ckp_path, lr=1e-5)
summary(model)

import gc
gc.collect()
torch.cuda.empty_cache()

In [ ]:
pl.seed_everything(43, True)
dev_path   = '/kaggle/input/cocked-data/dev'
test_path  = '/kaggle/input/cocked-data/test'
train_path = '/kaggle/input/cocked-data/train'
labeld_valid = '/kaggle/input/cocked-data/labeld_valid'

In [ ]:
train = list(map(lambda x: os.path.join(train_path, x), os.listdir(train_path)))
labeld_valid = list(map(lambda x: os.path.join(labeld_valid, x), os.listdir(labeld_valid)))
train = labeld_valid + labeld_valid + labeld_valid  + labeld_valid + labeld_valid + train[:-9] + labeld_valid + labeld_valid  + labeld_valid + labeld_valid
random.shuffle(train)
train = np.array(train)
dev = np.array(list(map(lambda x: os.path.join(dev_path, x), os.listdir(dev_path))))
test = np.array(list(map(lambda x: os.path.join(test_path, x), os.listdir(test_path))))

labeld_valid = np.array(labeld_valid)

In [ ]:
batch_size = 4

train = Data(train, batch_size, 1000, 300)
train = DataLoader(train, batch_size=None, shuffle=True, num_workers=2)

dev = Data(dev, batch_size, 1000, 300)
dev = DataLoader(dev, batch_size=None, shuffle=False, num_workers=2)

test = Data(test, batch_size, 1000, 300)
test = DataLoader(test, batch_size=None, shuffle=False, num_workers=2)

labeld_valid = Data(labeld_valid, batch_size, 1000, 300)
labeld_valid = DataLoader(labeld_valid, batch_size=None, shuffle=False, num_workers=2)

In [ ]:
gc.collect()
torch.cuda.empty_cache()
trainer = pl.Trainer(accelerator='auto', devices='auto',
                     max_epochs=200,
#                      max_time='00:11:00:00',
                     strategy='auto',
                     num_sanity_val_steps=2,
#                      limit_train_batches=10,

                     log_every_n_steps=25,
                     callbacks=[ckp, progress_bar],
                     accumulate_grad_batches=1, #precision='16-mixed',
                     gradient_clip_val=0.5,
                     sync_batchnorm=True,
                     logger=True,
                     enable_model_summary=True, enable_checkpointing=True, #benchmark=True,
                     default_root_dir=os.getcwd())


trainer.fit(model, labeld_valid, labeld_valid) #, ckpt_path='lightning_logs/version_0/checkpoints/epoch=5-step=504.ckpt')

In [ ]:
# !rm -r lightning_logs/

In [ ]:
new_ckp = 'lightning_logs/version_0/checkpoints/last.ckpt'
new_model = EncoderDecoder4Summary(model_name, bert_tokenizer.pad_token_id, 1000, ckp=new_ckp, lr=1e-4)


In [ ]:
def generate(document, model, pre, tok, max_lenght=511):
    model.eval().cuda()
    pre_doc = pre.preprocess(document)
    pre_doc = tok(pre_doc, return_attention_mask=True, padding='max_length', max_length=1000, truncation=True, return_tensors='pt')
    seq_in  = dict()
    seq_in['input_ids']  = pre_doc.input_ids.cuda()
    seq_in['attention_mask'] = pre_doc.attention_mask.cuda()
    seq_in['token_type_ids'] = pre_doc.token_type_ids.cuda()
    
    output_ids = torch.ones((1, 1)).long().cuda() * tok.cls_token_id
    with torch.no_grad():
        context  = model.encoder(**seq_in).last_hidden_state
        for i in range(max_lenght):
            next_ids = model.decoder(output_ids, encoder_hidden_states=context, encoder_attention_mask=seq_in['attention_mask'], output_attentions=False)
            next_ = torch.multinomial(next_ids.logits[0, -1, :].softmax(-1), 1)
            next_ = torch.unsqueeze(next_, 0)
            output_ids = torch.concat([output_ids, next_], 1)

            if tok.sep_token_id == next_[0, 0]:
                break
                
    output = tok.batch_decode(output_ids.cpu(), skip_special_tokens=True)[0]
    output = pre.unpreprocess(output)
    return output#, next_ids.cross_attentions

In [ ]:
new_test = pd.read_csv('/kaggle/input/arabic-text-summarization/data/labeld_valid.csv').values


In [ ]:
result = {'example_id':[], 'summary':[], 'truth': []}

for i in tqdm(range(len(new_test))):
    title, doc, summ = new_test[i]
    
    pre = generate(doc, new_model, bert_prep, bert_tokenizer)
    
    result['example_id'].append(i)
    result['summary'].append(pre)
    result['truth'].append(summ)

In [ ]:
avg_rough(result['summary'], result['truth'])

In [ ]:
result['summary'][4]

In [ ]:
result['truth'][4]

In [4]:
# !pip install git+https://github.com/ARBML/rouge_score_ar
!pip install accelerate -U

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.6/227.6 kB 8.5 MB/s eta 0:00:00
  Attempting uninstall: accelerate
    Found existing installation: accelerate 0.12.0
    Uninstalling accelerate-0.12.0:
      Successfully uninstalled accelerate-0.12.0


In [7]:
import pandas as pd
import tensorflow as tf
from rouge_score import rouge_scorer

In [ ]:
validation = pd.read_csv('/kaggle/input/t5-ar-finetune/data/dev/labeld_valid.csv')

In [1]:
from transformers import pipeline
from transformers import TFMT5ForConditionalGeneration, MT5Tokenizer


/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/__init__.py:98: UserWarning: unable to load libtensorflow_io_plugins.so: unable to open file: libtensorflow_io_plugins.so, from paths: ['/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/libtensorflow_io_plugins.so']
caused by: ['/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/libtensorflow_io_plugins.so: undefined symbol: _ZN3tsl6StatusC1EN10tensorflow5error4CodeESt17basic_string_viewIcSt11char_traitsIcEENS_14SourceLocationE']
  warnings.warn(f"unable to load libtensorflow_io_plugins.so: {e}")
/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/__init__.py:104: UserWarning: file system plugins are not loaded: unable to open file: libtensorflow_io.so, from paths: ['/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/libtensorflow_io.so']
caused by: ['/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/libtensorflow_io.so: undefined symbol: _ZTVN10tenso

In [2]:
model_name = '/kaggle/input/t5-ar-finetune/sebay/t5_ar'

# summarizer = pipeline("summarization", model=model_name, tokenizer=model_name, framework="tf")

tokenizer = MT5Tokenizer.from_pretrained(model_name)
        
model = TFMT5ForConditionalGeneration.from_pretrained(model_name)

/opt/conda/lib/python3.10/site-packages/keras/initializers/initializers.py:120: UserWarning: The initializer RandomNormal is unseeded and being called multiple times, which will return identical values each time (even if the initializer is unseeded). Please update your code to provide a seed to the initializer, or avoid using the same initalizer instance more than once.
  warnings.warn(
All model checkpoint layers were used when initializing TFMT5ForConditionalGeneration.

All the layers of TFMT5ForConditionalGeneration were initialized from the model checkpoint at /kaggle/input/t5-ar-finetune/sebay/t5_ar.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFMT5ForConditionalGeneration for predictions without further training.


In [3]:
model.save_pretrained("sebay/t5_ar_torch", safetensors=True) 
tokenizer.save_pretrained("sebay/t5_ar_torch") 

('sebay/t5_ar_torch/tokenizer_config.json',
 'sebay/t5_ar_torch/special_tokens_map.json',
 'sebay/t5_ar_torch/spiece.model',
 'sebay/t5_ar_torch/added_tokens.json')

In [4]:
del model, tokenizer

In [5]:
import gc
gc.collect()

70

In [2]:
import torch
from transformers import MT5ForConditionalGeneration, MT5Tokenizer
from accelerate import init_empty_weights, infer_auto_device_map


/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/__init__.py:98: UserWarning: unable to load libtensorflow_io_plugins.so: unable to open file: libtensorflow_io_plugins.so, from paths: ['/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/libtensorflow_io_plugins.so']
caused by: ['/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/libtensorflow_io_plugins.so: undefined symbol: _ZN3tsl6StatusC1EN10tensorflow5error4CodeESt17basic_string_viewIcSt11char_traitsIcEENS_14SourceLocationE']
  warnings.warn(f"unable to load libtensorflow_io_plugins.so: {e}")
/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/__init__.py:104: UserWarning: file system plugins are not loaded: unable to open file: libtensorflow_io.so, from paths: ['/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/libtensorflow_io.so']
caused by: ['/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/libtensorflow_io.so: undefined symbol: _ZTVN10tenso

In [3]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [29]:
!rm -r sebay/t5_ar_torch

In [1]:
!ls sebay/t5_ar_torch

config.json		pytorch_model.bin	 spiece.model
generation_config.json	special_tokens_map.json  tokenizer_config.json


In [4]:
import torch
from transformers import MT5ForConditionalGeneration, MT5Tokenizer
from accelerate import init_empty_weights, infer_auto_device_map

model_name = 'sebay/t5_ar_torch'
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# summarizer = pipeline("summarization", model=model_name, tokenizer=model_name, framework="tf")

tokenizer = MT5Tokenizer.from_pretrained(model_name)

model = MT5ForConditionalGeneration.from_pretrained(model_name, from_tf=True) #,
model.save_pretrained("sebay/t5_ar_torch",) 
tokenizer.save_pretrained("sebay/t5_ar_torch") 

# model = MT5ForConditionalGeneration.from_pretrained(model_name, torch_dtype=torch.float16)
# model.to(device);


In [5]:
model.device

device(type='cpu')

In [5]:
# model.to(device);

In [1]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
from arabert.preprocess import ArabertPreprocessor

model_name="abdalrahmanshahrour/arabartsummarization"
preprocessor = ArabertPreprocessor(model_name="")

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
pipeline = pipeline("text2text-generation",model=model,tokenizer=tokenizer)

text = "شهدت مدينة طرابلس، مساء أمس الأربعاء، احتجاجات شعبية وأعمال شغب لليوم الثالث على التوالي، وذلك بسبب تردي الوضع المعيشي والاقتصادي. واندلعت مواجهات عنيفة وعمليات كر وفر ما بين الجيش اللبناني والمحتجين استمرت لساعات، إثر محاولة فتح الطرقات المقطوعة، ما أدى إلى إصابة العشرات من الطرفين."
text = preprocessor.preprocess(text)

result = pipeline(text,
            pad_token_id=tokenizer.eos_token_id,
            num_beams=3,
            repetition_penalty=3.0,
            max_length=200,
            length_penalty=1.0,
            no_repeat_ngram_size = 3)[0]['generated_text']

print(result)

/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/__init__.py:98: UserWarning: unable to load libtensorflow_io_plugins.so: unable to open file: libtensorflow_io_plugins.so, from paths: ['/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/libtensorflow_io_plugins.so']
caused by: ['/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/libtensorflow_io_plugins.so: undefined symbol: _ZN3tsl6StatusC1EN10tensorflow5error4CodeESt17basic_string_viewIcSt11char_traitsIcEENS_14SourceLocationE']
  warnings.warn(f"unable to load libtensorflow_io_plugins.so: {e}")
/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/__init__.py:104: UserWarning: file system plugins are not loaded: unable to open file: libtensorflow_io.so, from paths: ['/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/libtensorflow_io.so']
caused by: ['/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/libtensorflow_io.so: undefined symbol: _ZTVN10tenso

تجددت الاشتباكات بين الجيش اللبناني ومحتجين في مدينة طرابلس شمالي لبنان.


In [6]:
# @tf.function
def generate_summary_pt(text):
    input_ids = tokenizer(
                        [text],
                        return_tensors="pt",
#                         padding="max_length",
                        truncation=True,
                        max_length=2000
                        )["input_ids"].to(device)

    output_ids = model.generate(
                                input_ids=input_ids,
                                max_new_tokens=900,
                                min_new_tokens= int(0.075 * len(text)),
                                no_repeat_ngram_size=4,
                                num_beams=25,
                                renormalize_logits =True,
                                temperature = 0.3,
                                top_k=100,
                                top_p = 0.7,
                                length_penalty=0.3,
                                )[0]
    summary = tokenizer.decode(
                                output_ids.cpu(),
                                skip_special_tokens=True,
                                clean_up_tokenization_spaces=True
                                )
    return summary


In [2]:
import pandas as pd
from tqdm import tqdm
validation = pd.read_csv('/kaggle/input/arabic-text-summarization/data/labeld_valid.csv')

In [15]:
validation = pd.read_csv('/kaggle/input/arabic-text-summarization/data/unlabeld_valid.csv')

In [16]:
validation

,example_id,document
0,0,وبعد أن ألقينا الضوء على أهم فتوحات بلاد الشام...
1,1,الفتوحات الإسلامية في عصر الخلفاء الراشدين .\n...
2,2,فتوحات بلاد شمال إفريقيا .\nقام المسلمون بالتو...
3,3,وبعد أن ألقينا الضوء على أهم فتوحات بلاد شمال ...
4,4,عزيزي الطالب / عزيزتي الطالبة ...\nبعد أن توقف...
...,...,...
267,267,وظهرت كثير من الدراسات قصرت اهتمامها على الأدو...
268,268,كانت الزراعة المصرية في هذه الحقبة تمر بعصرها ...
269,269,حشدت فرنسا قواتها وأعلنت الحرب في 19 يوليو أي ...
270,270,تعتبر قناة السويس هي من أهم الطرق الملاحية الت...


In [14]:
!rm rouge_score_ar/

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
rm: cannot remove 'rouge_score_ar/': Is a directory


In [19]:
result = {'example_id':[], 'summary': []}

for row in tqdm(validation.values):
    result['example_id'].append(row[0])
#     generated_summary = generate_summary_pt(row[1])
    generated_summary = preprocessor.preprocess(row[1])

    generated_summary = pipeline(generated_summary,
                pad_token_id=tokenizer.eos_token_id,
                num_beams=3,
                min_new_tokens=int(0.065 * len(generated_summary)),
                repetition_penalty=3.0,
                max_length=200,
                length_penalty=1.0,
                no_repeat_ngram_size = 3)[0]['generated_text']
    result['summary'].append(generated_summary)
    

100%|██████████| 272/272 [26:49<00:00,  5.92s/it]


In [20]:
final_submission = pd.DataFrame.from_records(result)

In [21]:
final_submission.to_json('predictions.jsonl', lines=True, orient='records', force_ascii=False)

In [23]:
final_submission.summary[10]

'بعد عشرة أيام من معركة بلاط الشهداء ( تور - بواتييه ) 114 ه - 732 م ، تمكن جيش المسلمين بقيادة " عبد الرحمن الغافقي " من فتح النصف الجنوبي لفرنسا، وترك حاميات عسكرية في المدن التي فتحها مما أضعف قوة جيشه قبيل لقائه مع " شارل مارتل " الذي استنجد به طالبا العون منه لاسترجاع دوقية أكوتين المسيحية من فرنسا. فما هي الأسباب التي أدت إلى هزيمة المسلمين في هذه المعركة ؟'

In [4]:
import pandas as pd
from tqdm import tqdm
validation = pd.read_csv('/kaggle/input/arabic-text-summarization/data/labeld_valid.csv')

In [5]:
validation

,example_id,document,summary
0,0,وتحت عنوان من الكارثة إلى التحدى يبدأ الكاتب ع...,يبدأ الكاتب عرض الكتاب الرابع تحت عنوان من الك...
1,1,ولم يعترف دبلوماسيو هاتين الدولتين بالعريضة ال...,دبلوماسيو الدولتين لم يعترفوا بالعريضة التي قا...
2,2,قامت ولاية حلب بعد اعلان الجنرال الفرنسي هنري ...,أعلن غورو الانتداب الفرنسي على سوريا لكي يعاقب...
3,3,دولة مصر العربيه هي ليست اي دوله وليست اي شعب ...,مصر هي أم البلاد، وقائدة العرب؛ فهي أرض بلاد ا...
4,4,السوريون يصرون على استقلال بلادهم : و مثلما رف...,الشعب السوري يصر على استقلال بلدهم من السيطرة ...
...,...,...,...
149,149,حزب الوفد سيحتفل بمئوية ثورة 1919 يوم 9 مارس ا...,احتفال مئوية ثورة 1919 كان من منطلق وطني ليس ح...
150,150,حيث أعلن مجلس قيادة الثورة في 18 يونيه 1953 قي...,مجلس قيادة الثورة أعلن عن قيام الجمهورية المصر...
151,151,وبرغم أن عبد الرحمن فهمي كان يضم في ذلك الجهاز...,ضم عبد الرحمن فهمي في الجهاز السري عدد كبير من...
152,152,ولم تقتصر مقومات بورسعيد كمدينة عالمية منذ نشأ...,امتدت بورسعيد لكي تشمل الطابع الثقافي للمدينة،...


In [62]:
# @tf.function
def generate_summary_pt(text):
    input_ids = tokenizer(
                        [text],
                        return_tensors="pt",
                        truncation=True,
                        max_length=2000
                        )["input_ids"].to(device)

    output_ids = model.generate(
                                input_ids=input_ids,
                                max_new_tokens=int(3*0.08 * len(text)),
                                penalty_alpha=0.6,
                                min_new_tokens= int(0.06 * len(text)),
                                no_repeat_ngram_size=4,
                                num_beams=25,
                                renormalize_logits =True,
                                temperature = 0.6,
#                                 top_k=3,
                                top_p = 0.9,
                                length_penalty=0.3,
                                )[0]
    summary = tokenizer.decode(
                                output_ids.cpu(),
                                skip_special_tokens=True,
                                clean_up_tokenization_spaces=True
                                )
    return summary


In [12]:
scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"])
scores = []
ll = len(validation.values)
for i, article in enumerate(validation.values):
    
    generated_summary = preprocessor.preprocess(article[1])
    rat = int(0.075 * len(article[1]))
    print("\r processing {} from {} with min token length {} and doc length {}".format(i, len(validation), rat, len(article[1].split())), end = "")
#     with tf.device('CPU'):
    generated_summary = pipeline(generated_summary,
                pad_token_id=tokenizer.eos_token_id,
                num_beams=3,
                min_new_tokens= int(0.075 * len(generated_summary)),
                repetition_penalty=3.0,
                max_length=200,
                length_penalty=1.0,
                no_repeat_ngram_size = 3)[0]['generated_text']
    new_score = scorer.score(article[2], generated_summary)
    scores.append(new_score)

    print("\tWith ROUGE scores: " , end= "")
    
    ss = {"rouge1": new_score["rouge1"].fmeasure ,
        "rouge2": new_score["rouge2"].fmeasure,
        "rougeL": new_score["rougeL"].fmeasure}
                   
   
    print(ss)
    
#     print('Ground Truth:')
#     print('\t\t', article[2])
#     print()
#     print('Model Prediction')
#     print('\t\t', generated_summary)
#     print()
#     print('='*20)

print()
print('='*50)
avg = {"rouge1": (sum(s["rouge1"].fmeasure for s in scores ) / ll),
        "rouge2": (sum(s["rouge2"].fmeasure for s in scores) / ll),
        "rougeL": (sum(s["rougeL"].fmeasure for s in scores) / ll)}
print('Average ROUGE scores', avg)

 processing 0 from 154 with min token length 134 and doc length 335	With ROUGE scores: {'rouge1': 0.2882882882882883, 'rouge2': 0.08181818181818182, 'rougeL': 0.16216216216216217}
 processing 1 from 154 with min token length 116 and doc length 277	With ROUGE scores: {'rouge1': 0.24203821656050953, 'rouge2': 0.09032258064516129, 'rougeL': 0.11464968152866241}
 processing 2 from 154 with min token length 143 and doc length 339	With ROUGE scores: {'rouge1': 0.24875621890547264, 'rouge2': 0.05025125628140704, 'rougeL': 0.1791044776119403}
 processing 3 from 154 with min token length 119 and doc length 282	With ROUGE scores: {'rouge1': 0.18584070796460175, 'rouge2': 0.0625, 'rougeL': 0.09734513274336283}
 processing 4 from 154 with min token length 107 and doc length 259	With ROUGE scores: {'rouge1': 0.21333333333333332, 'rouge2': 0.05405405405405405, 'rougeL': 0.1333333333333333}
 processing 5 from 154 with min token length 154 and doc length 368	With ROUGE scores: {'rouge1': 0.20883534136

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ /tmp/ipykernel_200/1284468113.py:10 in <module>                                                  │
│                                                                                                  │
│ [Errno 2] No such file or directory: '/tmp/ipykernel_200/1284468113.py'                          │
│                                                                                                  │
│ /opt/conda/lib/python3.10/site-packages/transformers/pipelines/text2text_generation.py:165 in    │
│ __call__                                                                                         │
│                                                                                                  │
│   162 │   │   │     ids of the generated text.                                                   │
│   163 │   │   """                                                                                │
│   164 │   │                                                                                      │
│ ❱ 165 │   │   result = super().__call__(*args, **kwargs)                                         │
│   166 │   │   if (                                                                               │
│   167 │   │   │   isinstance(args[0], list)                                                      │
│   168 │   │   │   and all(isinstance(el, str) for el in args[0])                                 │
│                                                                                                  │
│ /opt/conda/lib/python3.10/site-packages/transformers/pipelines/base.py:1119 in __call__          │
│                                                                                                  │
│   1116 │   │   │   │   )                                                                         │
│   1117 │   │   │   )                                                                             │
│   1118 │   │   else:                                                                             │
│ ❱ 1119 │   │   │   return self.run_single(inputs, preprocess_params, forward_params, postproces  │
│   1120 │                                                                                         │
│   1121 │   def run_multi(self, inputs, preprocess_params, forward_params, postprocess_params):   │
│   1122 │   │   return [self.run_single(item, preprocess_params, forward_params, postprocess_par  │
│                                                                                                  │
│ /opt/conda/lib/python3.10/site-packages/transformers/pipelines/base.py:1126 in run_single        │
│                                                                                                  │
│   1123 │                                                                                         │
│   1124 │   def run_single(self, inputs, preprocess_params, forward_params, postprocess_params):  │
│   1125 │   │   model_inputs = self.preprocess(inputs, **preprocess_params)                       │
│ ❱ 1126 │   │   model_outputs = self.forward(model_inputs, **forward_params)                      │
│   1127 │   │   outputs = self.postprocess(model_outputs, **postprocess_params)                   │
│   1128 │   │   return outputs                                                                    │
│   1129                                                                                           │
│                                                                                                  │
│ /opt/conda/lib/python3.10/site-packages/transformers/pipelines/base.py:1025 in forward           │
│                                                                                                  │
│   1022 │   │   │   │   inference_context = self.get_inference_context()                          │
│   1023 │   │   │   │   with inference_context():           

In [ ]:
avg = {"rouge1": (sum(s["rouge1"].fmeasure for s in scores ) / 154),
        "rouge2": (sum(s["rouge2"].fmeasure for s in scores) / 154),
        "rougeL": (sum(s["rougeL"].fmeasure for s in scores) / 154)}
print('Average ROUGE scores', avg)

In [ ]:
 sum(s["rouge1"].fmeasure for s in scores )/154